# 暫停交易：個股尚未恢復

從 rebuild 的 `trading_halt` 讀檔，篩出**個股**有停牌紀錄但 `resume_trading_date` 為空的列。

（`Y9999` 為加權指數休市日，不算個股。）


In [1]:
import sys
import os

_here = os.path.abspath(os.getcwd())
if _here not in sys.path:
    sys.path.insert(0, _here)

from parse import read_tej_txt, one_txt
import Trading_Halt
import polars as pl


In [2]:
# Windows: C:\Users\Ultra7\TEJ_data\... → WSL: /mnt/c/Users/Ultra7/TEJ_data/...
rebuild_root = "/mnt/c/Users/Ultra7/TEJ_data/rebuild_20150601~20260531"
trading_halt_dir = os.path.join(rebuild_root, "trading_halt")


def load_trading_halt(folder: str) -> pl.DataFrame:
    """讀取 folder 內唯一 .txt，回傳 Trading_Halt keep 欄位。"""
    path = one_txt(folder)
    return (
        read_tej_txt(path, Trading_Halt.TRADING_HALT_SCHEMA)
        .select(Trading_Halt.keep_col)
    )


trading_halt_df = load_trading_halt(trading_halt_dir)
print(f"檔案: {one_txt(trading_halt_dir)}")
print(f"總筆數: {trading_halt_df.height}")
trading_halt_df.head()


檔案: /mnt/c/Users/Ultra7/TEJ_data/rebuild_20150601~20260531/trading_halt/20260622013657.txt
總筆數: 1510


symbol,start_halt,resume_trading_date,reason
str,date,date,str
"""5466""",2015-06-05,2015-06-17,"""合併下市"""
"""910708""",2015-06-18,2015-06-22,"""自行申請"""
"""Y9999""",2015-06-19,null,"""端午節補假"""
"""Y9999""",2015-07-10,null,"""昌鴻颱風停班"""
"""6165""",2015-08-10,2015-08-24,"""減資"""


In [3]:
# 個股：排除指數 Y9999；resume 空＝尚未恢復
open_halt_df = trading_halt_df.filter(
    (pl.col("symbol") != "Y9999")
    & pl.col("start_halt").is_not_null()
    & pl.col("resume_trading_date").is_null()
)

print(f"個股尚未恢復: {open_halt_df.height} 筆 / {open_halt_df['symbol'].n_unique()} 檔")
print(open_halt_df.group_by("reason").len().sort("len", descending=True))
open_halt_df.sort("start_halt")


個股尚未恢復: 2 筆 / 2 檔
shape: (1, 2)
┌──────────┬─────┐
│ reason   ┆ len │
│ ---      ┆ --- │
│ str      ┆ u32 │
╞══════════╪═════╡
│ 違規財報 ┆ 2   │
└──────────┴─────┘


symbol,start_halt,resume_trading_date,reason
str,date,date,str
"""1589""",2026-04-07,null,"""違規財報"""
"""4804""",2026-04-14,null,"""違規財報"""
